# CFPB Complaint Routing & Trend Detection — Phase 4: Trend & Anomaly Detection
**Goal:** spot unusual spikes and emerging problems in complaint volumes early, so a complaints team (or a regulator) can react before an issue snowballs.

The routing models in Phases 2–3 need narratives, which the CFPB stopped publishing in Sept 2026. **Trend detection doesn't**: the structured fields (date, product, issue, company) are still published live, so this monitor runs on **current data**.

1. Load the **live** CFPB export (structured fields) and build weekly complaint counts per team, issue and company
2. A robust spike detector (rolling median + MAD), **validated by injecting synthetic spikes** to measure recall vs false-alarm rate
3. Run it across ~thousands of series and rank the alerts
4. **Emerging issues / companies:** recent 8 weeks vs the previous 26 weeks (Poisson rate test)
5. **What are people complaining about?** NMF topics on the archived narratives and which topics are rising

Runs on **Kaggle Notebooks** (CPU, Internet on).

## 1. Setup

In [ ]:
import os, re, time, json, gc, zipfile, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
SEED = 42; rng = np.random.default_rng(SEED)
OUT_DIR = "/kaggle/working"; TMP = "/kaggle/temp"; os.makedirs(TMP, exist_ok=True)
START = pd.Timestamp("2020-01-01")

PRODUCT_MAP = {
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
    "Credit reporting or other personal consumer reports": "Credit reporting",
    "Credit reporting": "Credit reporting",
    "Debt collection": "Debt collection", "Debt or credit management": "Debt collection",
    "Credit card or prepaid card": "Credit card", "Credit card": "Credit card",
    "Prepaid card": "Prepaid / money transfer",
    "Money transfer, virtual currency, or money service": "Prepaid / money transfer",
    "Money transfers": "Prepaid / money transfer", "Virtual currency": "Prepaid / money transfer",
    "Checking or savings account": "Bank account", "Bank account or service": "Bank account",
    "Mortgage": "Mortgage", "Vehicle loan or lease": "Vehicle loan", "Consumer Loan": "Vehicle loan",
    "Student loan": "Student loan",
    "Payday loan, title loan, or personal loan": "Personal / payday loan",
    "Payday loan, title loan, personal loan, or advance loan": "Personal / payday loan",
    "Payday loan": "Personal / payday loan",
}

## 2. Live structured data → weekly counts
We stream the official export (~5 GB unzipped) in chunks, keep 2020 onward, and aggregate on the fly to **weekly counts per (team, issue, company)** — the full table never sits in memory. If the live export is unreachable, we fall back to the archived snapshot (Aug 2026).

**Reporting lag:** complaints show up in the public database with a delay, so the newest weeks look artificially low. We detect and drop these immature weeks automatically — otherwise every series would appear to be *falling*.

In [ ]:
LIVE_URL = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
LIVE_ZIP = f"{TMP}/complaints.csv.zip"
COLS = {"Date received": "date", "Product": "product", "Issue": "issue", "Company": "company"}

def week_counts(chunk):
    chunk = chunk[chunk["date"] >= START]
    chunk = chunk.assign(week=chunk["date"].dt.to_period("W-SUN").dt.start_time,
                         team=chunk["product"].map(PRODUCT_MAP).fillna("Other"))
    return chunk.groupby(["week", "team", "issue", "company"], observed=True).size()

t0 = time.time(); parts, source, unmapped = [], None, pd.Series(dtype=int)
try:
    if not os.path.exists(LIVE_ZIP):
        urllib.request.urlretrieve(LIVE_URL, LIVE_ZIP)
    with zipfile.ZipFile(LIVE_ZIP) as zf, zf.open(zf.namelist()[0]) as f:
        for chunk in pd.read_csv(f, usecols=list(COLS), chunksize=500_000, dtype=str):
            chunk = chunk.rename(columns=COLS)
            chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
            unmapped = unmapped.add(chunk.loc[~chunk["product"].isin(PRODUCT_MAP), "product"].value_counts(), fill_value=0)
            parts.append(week_counts(chunk))
    source = "live CFPB export"
except Exception as e:
    print("live export unavailable -> archived snapshot:", repr(e)[:200])
    from huggingface_hub import hf_hub_download
    import pyarrow.parquet as pq
    PQ = hf_hub_download("vivekkopthsd/cfpb-consumer-complaints", "complaints.parquet", repo_type="dataset", local_dir=TMP)
    for batch in pq.ParquetFile(PQ).iter_batches(batch_size=500_000, columns=["date_received", "product", "issue", "company"]):
        chunk = batch.to_pandas().rename(columns={"date_received": "date"})
        chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
        parts.append(week_counts(chunk))
    source = "archived snapshot (Aug 2026)"

wk = pd.concat(parts).groupby(level=[0, 1, 2, 3], observed=True).sum().rename("n").reset_index()
del parts; gc.collect()
# Reporting lag: complaints appear in the public database with a delay, so the newest weeks are incomplete.
# Keep weeks whose total is >= 85% of the typical level 5-16 weeks back.
tot = wk.groupby("week")["n"].sum().sort_index()
ref = tot.iloc[-16:-4].median()
last_full_week = tot.index[-1]
while tot[last_full_week] < 0.85 * ref:
    print(f"  dropping immature week {last_full_week.date()}: {tot[last_full_week]:,} complaints = {tot[last_full_week]/ref:.0%} of normal")
    last_full_week -= pd.Timedelta(weeks=1)
wk = wk[wk["week"] <= last_full_week]
print(f"source: {source} | {wk['n'].sum():,} complaints {wk['week'].min().date()} -> {last_full_week.date()} "
      f"| {len(wk):,} week x team x issue x company rows | {time.time()-t0:.0f}s")
if len(unmapped):
    print("products mapped to 'Other':", unmapped.sort_values(ascending=False).head(5).to_dict())

In [ ]:
team_w = wk.pivot_table(index="week", columns="team", values="n", aggfunc="sum", fill_value=0)
fig, ax = plt.subplots(figsize=(12, 5))
team_w.drop(columns=[c for c in ["Other"] if c in team_w]).plot(ax=ax, lw=1.2)
ax.set(title=f"Weekly complaints by team ({source})", ylabel="complaints / week", xlabel="", yscale="log")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.tight_layout(); plt.show()
team_w.tail(4).T.astype(int)

## 3. The spike detector
For each weekly series we compare this week to the **previous 12 weeks**:
- baseline = rolling **median** (robust to past spikes), spread = 1.4826 × rolling **MAD**
- the spread is floored at √baseline (Poisson noise) so quiet series don't explode on tiny changes
- alert when `z = (count − baseline) / spread ≥ Z`, the count is at least `MIN_COUNT`, and at least 1.5× the baseline

Choosing **Z** needs evidence, so first we test the detector on synthetic spikes.

In [ ]:
W, MIN_HIST = 12, 8

def robust_z(y):
    """y: 1-D array of weekly counts -> (z, baseline) using only past weeks."""
    s = pd.Series(y, dtype=float)
    past = s.shift(1)
    base = past.rolling(W, min_periods=MIN_HIST).median()
    mad = (past - base).abs().rolling(W, min_periods=MIN_HIST).median() * 1.4826
    spread = np.maximum(np.maximum(mad, np.sqrt(base.clip(lower=1))), 1.0)
    return ((s - base) / spread).values, base.values

def detect(y, Z, min_count=20, min_ratio=1.5):
    z, base = robust_z(y)
    return (z >= Z) & (np.asarray(y) >= min_count) & (np.asarray(y) >= min_ratio * np.nan_to_num(base, nan=np.inf)), z, base

def dense(df, keys):
    """weekly matrix (rows = series, cols = all weeks, zeros filled)"""
    m = df.groupby(keys + ["week"], observed=True)["n"].sum().unstack("week", fill_value=0)
    return m.reindex(columns=pd.date_range(df["week"].min(), df["week"].max(), freq="W-MON"), fill_value=0)

issue_m = dense(wk, ["team", "issue"])
print("team x issue series:", issue_m.shape)

### 3a. Synthetic-spike evaluation
Take stable, reasonably busy series (team × issue, mean ≥ 20/week), inject a spike of size *m × baseline* into random weeks, and measure:
- **recall** = share of injected spikes the detector catches
- **alert rate on untouched weeks** = how often it fires on normal data (an upper bound on false alarms, since real spikes exist too)

In [ ]:
busy = issue_m[issue_m.mean(axis=1) >= 20]
Zs, mults = [3, 4, 5, 6], [1.5, 2.0, 3.0]
rec = {(Z, m): [] for Z in Zs for m in mults}; fa = {Z: [] for Z in Zs}
for _, row in busy.iterrows():
    y = row.values.astype(float)
    _, base = robust_z(y)
    for Z in Zs:
        fa[Z].append(np.nanmean(detect(y, Z)[0][MIN_HIST + 1:]))
    weeks = rng.choice(np.arange(MIN_HIST + 1, len(y)), size=min(8, len(y) - MIN_HIST - 1), replace=False)
    for m in mults:
        for t in weeks:
            if not np.isfinite(base[t]) or base[t] < 5:
                continue
            y2 = y.copy(); y2[t] = y[t] + (m - 1) * base[t]          # spike of size m x baseline
            for Z in Zs:
                rec[(Z, m)].append(bool(detect(y2, Z)[0][t]))
evalt = pd.DataFrame({f"recall @ {m}x": [np.mean(rec[(Z, m)]) for Z in Zs] for m in mults}, index=[f"Z={Z}" for Z in Zs])
evalt["alert rate (untouched weeks)"] = [np.mean(fa[Z]) for Z in Zs]
evalt["alerts / series / year"] = evalt["alert rate (untouched weeks)"] * 52
print(f"{len(busy)} busy series, {len(rec[(Zs[0], mults[0])]):,} injected spikes per setting")
evalt.round(3)

In [ ]:
Z_STAR = next((Z for Z in Zs if evalt.loc[f"Z={Z}", "alert rate (untouched weeks)"] <= 0.01), Zs[-1])
fig, ax = plt.subplots(figsize=(7, 4))
for Z in Zs:
    ax.plot(mults, [np.mean(rec[(Z, m)]) for m in mults], marker="o", label=f"Z={Z} ({evalt.loc[f'Z={Z}', 'alerts / series / year']:.1f} alerts/series/yr)")
ax.set(title="Detector recall on injected spikes", xlabel="spike size (x baseline)", ylabel="recall", ylim=(0, 1.02))
ax.legend(); plt.show()
print(f"chosen threshold Z* = {Z_STAR}: the lowest Z with <=1% alert rate on untouched weeks")

## 4. Running the monitor
Three levels: **team** (9 series), **team × issue**, and **company** (top 300 companies by volume).

In [ ]:
company_m = dense(wk, ["company"])
company_m = company_m.loc[company_m.sum(axis=1).sort_values(ascending=False).index[:300]]
team_m = dense(wk, ["team"])

def alerts_for(mat, level):
    rows = []
    for key, row in mat.iterrows():
        y = row.values.astype(float); flag, z, base = detect(y, Z_STAR)
        for t in np.where(flag)[0]:
            rows.append({"level": level, "series": " | ".join(key) if isinstance(key, tuple) else key,
                         "week": mat.columns[t].date(), "count": int(y[t]), "baseline": round(float(base[t]), 1),
                         "ratio": round(float(y[t] / max(base[t], 1)), 2), "z": round(float(z[t]), 1)})
    return rows
alerts = pd.DataFrame(alerts_for(team_m, "team") + alerts_for(issue_m, "team x issue") + alerts_for(company_m, "company"))
alerts["week"] = pd.to_datetime(alerts["week"])
print(alerts.groupby("level").size().rename("alerts since 2020").to_string())
recent = alerts[alerts["week"] >= last_full_week - pd.Timedelta(weeks=26)].sort_values("z", ascending=False)
print("\nLargest alerts since 2020:")
display(alerts.sort_values("z", ascending=False).head(10))
print(f"\nAlerts in the last 26 weeks (to {last_full_week.date()}):")
recent.head(15)

In [ ]:
def plot_series(mat, key, title, ax):
    y = mat.loc[key].values.astype(float); flag, z, base = detect(y, Z_STAR)
    x = mat.columns
    ax.plot(x, y, lw=1, label="weekly complaints"); ax.plot(x, base, lw=1, ls="--", label="baseline (12-wk median)")
    ax.scatter(x[flag], y[flag], color="crimson", zorder=3, s=25, label="alert")
    ax.set_title(title[:70], fontsize=10); ax.legend(fontsize=8)
show = recent.drop_duplicates("series").head(3)
fig, axes = plt.subplots(len(show), 1, figsize=(12, 3.2 * len(show)), squeeze=False)
for ax, (_, r) in zip(axes[:, 0], show.iterrows()):
    mat = {"team": team_m, "team x issue": issue_m, "company": company_m}[r.level]
    key = tuple(r.series.split(" | ")) if r.level == "team x issue" else r.series
    plot_series(mat, key, f"{r.level}: {r.series}", ax)
plt.tight_layout(); plt.show()

## 5. Emerging issues & companies
Spikes catch sudden jumps; **emerging** problems grow over weeks. For each series compare the **last 8 weeks** with the **previous 26 weeks**:
`expected = 8 × (weekly mean over the previous 26)`, `z = (observed − expected) / (√8 × weekly s.d.)`.
A pure Poisson z (√expected) badly overstates significance for big, noisy series — a 1.08× wobble in credit reporting (600k complaints) would score z ≈ 60 — so we use each series' own week-to-week spread (never below Poisson), and keep series with ≥ 50 recent complaints and ≥ 1.2× growth.

In [ ]:
def emerging(mat, recent_w=8, prev_w=26, min_obs=50, min_ratio=1.2):
    recent = mat.iloc[:, -recent_w:]; prev = mat.iloc[:, -(recent_w + prev_w):-recent_w]
    obs, mu = recent.sum(axis=1), prev.mean(axis=1)
    exp = recent_w * mu
    # overdispersion-aware noise: use the series' own week-to-week spread (never less than Poisson)
    sd = np.sqrt(recent_w) * np.maximum(prev.std(axis=1), np.sqrt(mu.clip(lower=1)))
    out = pd.DataFrame({"recent 8 wks": obs, "expected": exp.round(1),
                        "ratio": (obs / exp.clip(lower=1)).round(2), "z": ((obs - exp) / sd).round(1)})
    return out[(out["recent 8 wks"] >= min_obs) & (out["ratio"] >= min_ratio)].sort_values("z", ascending=False)
em_issue, em_company = emerging(issue_m), emerging(company_m)
em_issue.index = [" | ".join(i) for i in em_issue.index]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (name, e) in zip(axes, [("team | issue", em_issue), ("company", em_company)]):
    top = e.head(10).iloc[::-1]
    ax.barh([s[:55] for s in top.index], top["ratio"], color="#C44E52")
    ax.axvline(1, color="grey", lw=1, ls="--"); ax.set(title=f"Fastest-growing {name}: last 8 wks vs prior 26", xlabel="rate ratio")
plt.tight_layout(); plt.show()
print("Top emerging issues:"); display(em_issue.head(10))
print("Top emerging companies:"); display(em_company.head(10))

## 6. What are people complaining about? Topic trends in the narratives
Structured "issue" labels are coarse and chosen by the consumer from a dropdown. The narratives say much more, so we learn **20 topics with NMF** on TF-IDF features of the archived narratives (2020 → Jul 2026), then track each topic's monthly share.
**Mix adjustment:** the archive's product mix changes sharply over time (credit-reporting narratives collapse in 2025–26), which would masquerade as topic change. We therefore compare topic shares *within* each team, re-weighted to the 2024 team mix.
This rebuilds the Phase 1 narrative sample (same seed).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq
PQ = hf_hub_download("vivekkopthsd/cfpb-consumer-complaints", "complaints.parquet", repo_type="dataset", local_dir=TMP)
parts = []
for batch in pq.ParquetFile(PQ).iter_batches(batch_size=250_000, columns=["date_received", "product", "narrative"]):
    chunk = batch.to_pandas()
    chunk["date_received"] = pd.to_datetime(chunk["date_received"], errors="coerce")
    chunk = chunk[chunk["narrative"].notna() & (chunk["narrative"].str.len() > 0) & (chunk["date_received"] >= START)]
    parts.append(chunk.sample(frac=0.15, random_state=SEED))
nar = pd.concat(parts, ignore_index=True); del parts; gc.collect()
nar["text"] = (nar["narrative"].str.replace(r"X{2,}", " ", regex=True)
                              .str.replace(r"\{\$[\d,.]+\}", " ", regex=True).str.replace(r"\s+", " ", regex=True))
nar = nar.drop_duplicates("text"); nar = nar[nar["text"].str.split().str.len() >= 10].reset_index(drop=True)
nar["month"] = nar["date_received"].dt.to_period("M").dt.start_time
nar["team"] = nar["product"].map(PRODUCT_MAP).fillna("Other")
print(f"{len(nar):,} narratives {nar['month'].min().date()} -> {nar['month'].max().date()}")

In [ ]:
t0 = time.time()
tf = TfidfVectorizer(max_features=30_000, min_df=20, max_df=0.5, stop_words="english", ngram_range=(1, 2),
                     sublinear_tf=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b", dtype=np.float32)
fit_idx = rng.choice(len(nar), size=min(120_000, len(nar)), replace=False)
X_fit = tf.fit_transform(nar["text"].iloc[fit_idx])
K = 20
nmf = NMF(n_components=K, init="nndsvda", max_iter=300, random_state=SEED).fit(X_fit)
vocab = np.array(tf.get_feature_names_out())
topics = pd.DataFrame({"top words": [", ".join(vocab[np.argsort(c)[-10:][::-1]]) for c in nmf.components_]})
topics.index = [f"T{k:02d}" for k in range(K)]
W_all = np.vstack([nmf.transform(tf.transform(nar["text"].iloc[i:i + 50_000])) for i in range(0, len(nar), 50_000)])
nar["topic"] = np.array(topics.index)[W_all.argmax(1)]
print(f"NMF fitted + applied in {time.time()-t0:.0f}s")
topics["share overall"] = nar["topic"].value_counts(normalize=True).reindex(topics.index).round(3)
topics

In [ ]:
raw = pd.crosstab(nar["month"], nar["topic"], normalize="index")
# Mix adjustment: the archive's product mix shifts a lot (credit-reporting narratives collapse in 2025-26),
# which drags topic shares with it. Re-weight within-team topic shares by the fixed 2024 team mix.
within = pd.crosstab([nar["month"], nar["team"]], nar["topic"], normalize="index")
w24 = nar.loc[nar["month"].dt.year == 2024, "team"].value_counts(normalize=True)
adj = {}
for m, g in within.groupby(level="month"):
    teams = g.index.get_level_values("team"); wts = w24.reindex(teams).fillna(0).values
    adj[m] = (g.values * wts[:, None]).sum(0) / max(wts.sum(), 1e-9)
share = pd.DataFrame(adj, index=within.columns).T.sort_index()
def change(tab):
    last6 = tab.iloc[-6:].mean(); base = tab[(tab.index >= "2024-01-01") & (tab.index < "2025-01-01")].mean()
    return base, last6
b_raw, l_raw = change(raw); b_adj, l_adj = change(share)
trend = pd.DataFrame({"share 2024": b_adj.round(3), "share last 6 mo (mix-adj.)": l_adj.round(3),
                      "change, mix-adjusted (pts)": ((l_adj - b_adj) * 100).round(1),
                      "change, raw (pts)": ((l_raw - b_raw) * 100).round(1), "top words": topics["top words"]}
                     ).sort_values("change, mix-adjusted (pts)", ascending=False)
q = share.groupby(share.index.to_period("Q")).mean()
fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap((q.T * 100).round(1), cmap="Blues", ax=ax, cbar_kws={"label": "% of narratives (mix-adjusted)"},
            yticklabels=[f"{t}: {', '.join(topics.loc[t, 'top words'].split(', ')[:4])}" for t in q.columns])
ax.set(title="Topic share by quarter, adjusted to the 2024 product mix", xlabel="quarter", ylabel=""); plt.tight_layout(); plt.show()
LEGAL = re.compile(r"\b(usc|section|sections|shall|states consumer|section states|act fcra|furnish|reinvestigation|person shall)\b")
template_topics = [t for t in topics.index if len(LEGAL.findall(topics.loc[t, "top words"])) >= 2]
print(f"template / legal-citation topics: {template_topics} -> {nar['topic'].isin(template_topics).mean():.1%} of narratives")
print("\nRising and falling topics (last 6 months vs 2024, mix-adjusted):")
pd.concat([trend.head(5), trend.tail(3)])

## 7. Save the monitor outputs

In [ ]:
alerts.to_csv(f"{OUT_DIR}/alerts.csv", index=False)
em_issue.head(50).to_csv(f"{OUT_DIR}/emerging_issues.csv"); em_company.head(50).to_csv(f"{OUT_DIR}/emerging_companies.csv")
trend.to_csv(f"{OUT_DIR}/topic_trends.csv")
summary = {"phase": 4, "source": source, "data_through": str(last_full_week.date()),
           "complaints_since_2020": int(wk["n"].sum()), "detector": {"window_weeks": W, "Z": Z_STAR},
           "synthetic_eval": evalt.round(4).reset_index().to_dict(orient="records"),
           "alerts_by_level": alerts.groupby("level").size().to_dict(),
           "top_recent_alerts": recent.head(10).astype(str).to_dict(orient="records"),
           "top_emerging_issues": em_issue.head(5).reset_index().astype(str).to_dict(orient="records"),
           "top_emerging_companies": em_company.head(5).reset_index().astype(str).to_dict(orient="records"),
           "rising_topics": trend.head(3).reset_index().astype(str).to_dict(orient="records")}
json.dump(summary, open(f"{OUT_DIR}/phase4_summary.json", "w"), indent=2, default=str)
print(json.dumps({k: summary[k] for k in ["source", "data_through", "complaints_since_2020", "detector", "alerts_by_level"]}, indent=1))

## Findings & next steps
*(summarised in the README after the run)*
- **Phase 5:** serve the router (Phase 3 model) behind an API and turn this monitor into a small dashboard that refreshes weekly from the live export.